# Buscape PT-BR Sentiment Analysis

## 1. Imports

In [1]:
import pickle
import string

import nltk
import unidecode
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load Data

In [14]:
dataset = pd.read_csv('../data/b2w_train.csv')
dataset.head()

,review_text,polarity,rating
0,Queria um computador para tarefas de rotina e ...,1.0,4
1,"Gostei muito do produto, só poderia ter alça p...",1.0,4
2,Muito bom produto produto bem feito..............,1.0,4
3,"O Conjunto é excelente, com variadas formas e ...",1.0,5
4,Essa chave é capaz de ds dar o torque correto...,1.0,5


## 3. Cleaning & Balancing

In [16]:
dataset["polarity_label"] = dataset["polarity"].map({0.0: "negative", 1.0: "positive"})

positive = dataset[dataset["polarity_label"] == "positive"].sample(28589, random_state=42)
negative = dataset[dataset["polarity_label"] == "negative"]
dataset = pd.concat([positive, negative]).reset_index(drop=True)

print(f"Dataset shape: {dataset.shape}")
print(dataset["polarity_label"].value_counts())


Dataset shape: (57178, 4)
polarity_label
positive    28589
negative    28589
Name: count, dtype: int64


## 4. Text Pre-processing

In [17]:
nltk.download("stopwords", quiet=True)
nltk.download("rslp", quiet=True)

puncts = list(string.punctuation)
stopwords = list(set([
    unidecode.unidecode(sw)
    for sw in nltk.corpus.stopwords.words("portuguese")
]))
stopwords_puncts = set(stopwords + puncts)

def preprocess(text: str) -> str:
    # 1. Gestion des valeurs nulles (NaN)
    if not isinstance(text, str) or text.strip() == "":
        return ""

    tokenizer = nltk.tokenize.WordPunctTokenizer()
    stemmer   = nltk.RSLPStemmer()
    
    tokens = tokenizer.tokenize(text)
    # Nettoyage (minuscules + suppression accents)
    tokens = [unidecode.unidecode(t.lower()) for t in tokens]
    
    # FILTRE CRUCIAL : On vérifie que 't' n'est pas vide ET pas dans les stopwords
    tokens = [t for t in tokens if t.strip() and t not in stopwords_puncts]
    
    # Stemming sécurisé
    tokens = [stemmer.stem(t) for t in tokens if len(t) > 0]
    
    return " ".join(tokens)

dataset["text_clean"] = dataset["review_text"].apply(preprocess)
dataset[["review_text", "text_clean", "polarity_label"]].head()


,review_text,text_clean,polarity_label
0,Gostei bastante. Joguei com meu filho de sete ...,gost bast jog filh set ano divert,positive
1,Produto de qualidade ótima. E a entrega foi mu...,produt qual otim entreg rap,positive
2,"Superou minhas expectativas,recebi antes do pr...",super expect receb ant praz estabelec ..,positive
3,possui a amioria das funcionalidades existente...,possu amior funcional exist boa cam bom audi,positive
4,"Não chegou muito rápido, mas o produto veio ce...",cheg rap produt vei cert bom val pen compr peq...,positive


## 5. Vectorization (TF-IDF + n-grams)

In [18]:
vectorizer = TfidfVectorizer(lowercase=False, max_features=100, ngram_range=(1, 2))
X = vectorizer.fit_transform(dataset["text_clean"])
y = dataset["polarity"]

print(f"Feature matrix shape: {X.shape}")


Feature matrix shape: (57178, 100)


## 6. Train / Test Split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.7,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")


Train size: 40024 | Test size: 17154


## 7. Training

In [20]:
model = LogisticRegression(solver="lbfgs", max_iter=1000)
model.fit(X_train, y_train)
print("Training complete.")


Training complete.


## 8. Evaluation

In [21]:
y_pred = model.predict(X_test)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy : 0.8735

Classification Report:
              precision    recall  f1-score   support

    negative       0.86      0.90      0.88      8577
    positive       0.89      0.85      0.87      8577

    accuracy                           0.87     17154
   macro avg       0.87      0.87      0.87     17154
weighted avg       0.87      0.87      0.87     17154

Confusion Matrix:
[[7708  869]
 [1301 7276]]


## 9. Save Model

In [22]:
pickle.dump(model,      open("../models/model.pkl", "wb"))
pickle.dump(vectorizer, open("../models/vectorizer.pkl", "wb"))
print("Model saved → ../models/model.pkl")
print("Vectorizer saved → ../models/vectorizer.pkl")


Model saved → ../models/model.pkl
Vectorizer saved → ../models/vectorizer.pkl
